In [1]:
# 데이터 처리 및 분석
import pandas as pd
import ast
import numpy as np
from datetime import datetime, timedelta
import warnings
import re

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
import scikit_posthocs as sp
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import platform

# ── 한글 폰트 설정 ──
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

# ── 출력 설정 ──
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
np.random.seed(42)

# 시드 설정
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("=" * 60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [ ]:
df_reviews = pd.read_csv('./data/lululemon_reviews_master_20260417.csv')

In [4]:
# ─── df_reviews 스키마 정리 ───────────────────────────────────────────────────

# 0. 한국 리뷰만 필터링 (locale == 'ko_KR')
df_reviews = df_reviews[df_reviews['locale'] == 'ko_KR'].copy()
print(f"ko_KR 필터 후: {df_reviews.shape[0]:,}건")

# 1. product_name 조인 (df에서 product_id 기준)
product_info = df[['product_id', 'product_name']].drop_duplicates('product_id')
df_reviews = df_reviews.merge(product_info, on='product_id', how='left')

# 2. 날짜 변환
df_reviews['collect_date'] = pd.to_datetime(df_reviews['collect_date'])
df_reviews['review_date']  = pd.to_datetime(df_reviews['date'])

# 3. 파생 컬럼
df_reviews['year']  = df_reviews['review_date'].dt.year.astype(int)
df_reviews['month'] = df_reviews['review_date'].dt.month.astype(int)

# 4. review_body = title + review_text 합치기 (title이 있는 경우만)
def combine_review(row):
    title = str(row['title']).strip() if pd.notna(row['title']) else ''
    body  = str(row['review_text']).strip() if pd.notna(row['review_text']) else ''
    if title:
        return f"{title} {body}".strip()
    return body

df_reviews['review_body'] = df_reviews.apply(combine_review, axis=1)

# 5. 컬럼명 매핑
df_reviews['helpful_count']   = df_reviews['helpful']
df_reviews['purchase_option'] = df_reviews['purchased_size']

# 6. 원본에 정보 없는 컬럼 → NaN
df_reviews['has_image']    = np.nan
df_reviews['user_height']  = np.nan
df_reviews['user_weight']  = np.nan
df_reviews['product_url']  = np.nan

# 7. platform 고정값
df_reviews['platform'] = '자사몰'

# 8. search_keyword: 이 데이터셋은 product_id 기준으로 수집 → product_id 사용
#    (다른 데이터셋은 product_name 또는 기타 텍스트가 키가 될 수 있음)
df_reviews['search_keyword'] = df_reviews['product_id']

# 9. user_height_group / user_weight_group → 원데이터 그대로 사용
df_reviews['user_height_group'] = df_reviews['height_group']
df_reviews['user_weight_group'] = df_reviews['weight_group']

# 10. 스키마 순서로 컬럼 재배치
schema_cols = [
    'collect_date', 'platform', 'search_keyword',
    'review_id', 'product_id', 'product_name',
    'review_date', 'year', 'month', 'review_body', 'rating',
    'helpful_count', 'has_image', 'purchase_option',
    'user_height', 'user_weight', 'user_height_group', 'user_weight_group',
    'product_url',
]
df_reviews = df_reviews[schema_cols]

# 11. 타입 지정
df_reviews['review_id']         = df_reviews['review_id'].astype(str)
df_reviews['product_id']        = df_reviews['product_id'].astype(str)
df_reviews['search_keyword']    = df_reviews['search_keyword'].astype(str)
df_reviews['rating']            = df_reviews['rating'].astype(int)
df_reviews['helpful_count']     = df_reviews['helpful_count'].astype(int)
df_reviews['has_image']         = df_reviews['has_image'].astype(float)
df_reviews['user_height']       = df_reviews['user_height'].astype(float)
df_reviews['user_weight']       = df_reviews['user_weight'].astype(float)
df_reviews['platform']          = df_reviews['platform'].astype('category')
df_reviews['purchase_option']   = df_reviews['purchase_option'].astype('category')
df_reviews['user_height_group'] = df_reviews['user_height_group'].astype('category')
df_reviews['user_weight_group'] = df_reviews['user_weight_group'].astype('category')

print(f"최종 shape: {df_reviews.shape}")
print()
print(df_reviews.dtypes)
print()
df_reviews.head(3)

ko_KR 필터 후: 11,819건
최종 shape: (11819, 19)

collect_date         datetime64[us]
platform                   category
search_keyword                  str
review_id                       str
product_id                      str
product_name                    str
review_date          datetime64[us]
year                          int64
month                         int64
review_body                     str
rating                        int64
helpful_count                 int64
has_image                   float64
purchase_option            category
user_height                 float64
user_weight                 float64
user_height_group          category
user_weight_group          category
product_url                 float64
dtype: object



,collect_date,platform,search_keyword,review_id,product_id,product_name,review_date,year,month,review_body,rating,helpful_count,has_image,purchase_option,user_height,user_weight,user_height_group,user_weight_group,product_url
0,2026-04-17,자사몰,prod11710026,384702382,prod11710026,메탈 벤트 테크 숏슬리브 셔츠,2026-04-16,2026,4,부드러운 옷 더운날 입기좋은 재질 부드러운 재질 땀냄새 나지 않는 재질 가격이 사...,5,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-04-17,자사몰,prod11710026,384590413,prod11710026,메탈 벤트 테크 숏슬리브 셔츠,2026-04-15,2026,4,같은옷 3개째 같은 색상의. 옷을 3개 구매 \n\n장점 : 오늘은 뭐입지 라는 고...,5,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-04-17,자사몰,prod11710026,384591246,prod11710026,메탈 벤트 테크 숏슬리브 셔츠,2026-04-15,2026,4,3점 두께감이있어서 호불호가있고 색감이화면과는 조금상이합니다. 불만족하였는데 리뷰...,3,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# ─── 결측치 비율 확인 ─────────────────────────────────────────────────────────

null_df = pd.DataFrame({
    '결측치 수': df_reviews.isnull().sum(),
    '결측치 비율(%)': (df_reviews.isnull().sum() / len(df_reviews) * 100).round(2),
})
null_df[null_df['결측치 수'] > 0].sort_values('결측치 비율(%)', ascending=False)

,결측치 수,결측치 비율(%)
has_image,11819,100.00
user_height,11819,100.00
user_weight,11819,100.00
product_url,11819,100.00
user_height_group,11537,97.61
user_weight_group,7551,63.89
purchase_option,7330,62.02
product_name,1527,12.92


In [6]:
# ─── 중복 확인 ───────────────────────────────────────────────────────────────

# review_id 기준 중복
dup_id = df_reviews.duplicated(subset='review_id', keep=False)
print(f"review_id 중복: {dup_id.sum()}건 ({dup_id.sum() / len(df_reviews) * 100:.2f}%)")

# 내용 기준 중복 (product_id + review_body)
dup_content = df_reviews.duplicated(subset=['product_id', 'review_body'], keep=False)
print(f"product_id + review_body 중복: {dup_content.sum()}건 ({dup_content.sum() / len(df_reviews) * 100:.2f}%)")

# 중복 있으면 샘플 확인
if dup_id.sum() > 0:
    print()
    print("▶ review_id 중복 샘플")
    display(df_reviews[dup_id].sort_values('review_id').head(6))

review_id 중복: 0건 (0.00%)
product_id + review_body 중복: 5건 (0.04%)


In [ ]:
# # ─── 파일 저장 ───────────────────────────────────────────────────────────────

# output_path = './data/lululemon_reviews_cleaned_20260425.csv'
# df_reviews.to_csv(output_path, index=False, encoding='utf-8-sig')
# print(f"저장 완료: {output_path} ({len(df_reviews):,}건)")

저장 완료: ./data/lululemon_reviews_cleaned_20260425.csv (11,819건)


In [8]:
df_reviews.shape

(11819, 19)

In [9]:
df_musinsa_reviews = pd.read_csv('./data/musinsa_lululemon_reviews.csv')
df_musinsa_reviews.head()

,collect_date,review_date,product_id,product_name,review_no,review_type,rating,review_text,author,user_height,user_weight,option,helpful,comment_count,has_image,image_urls,survey_size,survey_width,survey_comfort,survey_instep,url
0,2026-04-21 22:57:48,2026-04-09,5733085,심리스 와이드 헤드밴드 SOGY VPOR,83547450,일반,4,룰루레몬 헤어밴드 좋습니다. 러닝할때 아주 잘 사용해요,산만한브라운정장,NaN,NaN,Solar Grey/Vapor · One Size,0,0,True,https://www.musinsa.com/data/estimate/5733085_...,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5733085#revie...
1,2026-04-21 22:57:48,2026-04-04,5733085,심리스 와이드 헤드밴드 SOGY VPOR,83449647,일반,5,땀이 많은 편이라 러닝 한번 하고나면 상의가 반쯤 젖을만큼 땀을 흘리는데 확실히 밴...,None27,NaN,NaN,Solar Grey/Vapor · One Size,0,0,False,NaN,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5733085#revie...
2,2026-04-21 22:57:48,2026-04-04,5733085,심리스 와이드 헤드밴드 SOGY VPOR,83439220,일반,5,안벗겨지고 깔끔하고 무난합니다. 땀도 잘 흡수하고 가볍고 빨리 마르고 착용감도 너어...,비범한제노바티켓,NaN,NaN,Solar Grey/Vapor · One Size,0,0,True,https://www.musinsa.com/data/estimate/5733085_...,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5733085#revie...
3,2026-04-21 22:57:48,2026-04-04,5733085,심리스 와이드 헤드밴드 SOGY VPOR,83189716,일반,5,벌써 여러 가지 색상으로 룰루레몬에 헤어밴드를 많이 가지고 있습니다. 매우 만족하며...,헬미온,NaN,NaN,Solar Grey/Vapor · One Size,0,0,True,https://www.musinsa.com/data/estimate/5733085_...,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5733085#revie...
4,2026-04-21 22:57:48,2026-04-01,5733085,심리스 와이드 헤드밴드 SOGY VPOR,83375899,일반,5,운동할 때 쓰려고 샀는데 진짜 너무 좋아요 재질도 너무 만족스럽네요,뭔상관ㅋ,NaN,NaN,Solar Grey/Vapor · One Size,0,0,False,NaN,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5733085#revie...


In [10]:
df_musinsa_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 1944 entries, 0 to 1943
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   collect_date    1944 non-null   str    
 1   review_date     1944 non-null   str    
 2   product_id      1944 non-null   int64  
 3   product_name    1944 non-null   str    
 4   review_no       1944 non-null   int64  
 5   review_type     1944 non-null   str    
 6   rating          1944 non-null   int64  
 7   review_text     1944 non-null   str    
 8   author          1944 non-null   str    
 9   user_height     1600 non-null   float64
 10  user_weight     1600 non-null   float64
 11  option          1944 non-null   str    
 12  helpful         1944 non-null   int64  
 13  comment_count   1944 non-null   int64  
 14  has_image       1944 non-null   bool   
 15  image_urls      1181 non-null   str    
 16  survey_size     0 non-null      float64
 17  survey_width    0 non-null      float64
 18 

In [11]:
# ─── df_musinsa_reviews 스키마 정리 ──────────────────────────────────────────

# 1. 날짜 변환
df_musinsa_reviews['collect_date'] = pd.to_datetime(df_musinsa_reviews['collect_date'])
df_musinsa_reviews['review_date']  = pd.to_datetime(df_musinsa_reviews['review_date'])

# 2. 파생 컬럼
df_musinsa_reviews['year']  = df_musinsa_reviews['review_date'].dt.year.astype(int)
df_musinsa_reviews['month'] = df_musinsa_reviews['review_date'].dt.month.astype(int)

# 3. 컬럼명 매핑
df_musinsa_reviews['review_id']       = df_musinsa_reviews['review_no']
df_musinsa_reviews['review_body']     = df_musinsa_reviews['review_text']
df_musinsa_reviews['helpful_count']   = df_musinsa_reviews['helpful']
df_musinsa_reviews['purchase_option'] = df_musinsa_reviews['option']
df_musinsa_reviews['product_url']     = df_musinsa_reviews['url']

# 4. has_image: bool → int (True=1, False=0)
df_musinsa_reviews['has_image'] = df_musinsa_reviews['has_image'].astype(int)

# 5. platform 고정값
df_musinsa_reviews['platform'] = '무신사'

# 6. search_keyword: product_id 기준으로 수집
df_musinsa_reviews['search_keyword'] = df_musinsa_reviews['product_id'].astype(str)

# 7. user_height_group: 실수값 → 5cm 구간 파생
height_bins   = [0, 150, 155, 160, 165, 170, 175, 180, 185, float('inf')]
height_labels = ['150cm 이하', '151-155cm', '156-160cm', '161-165cm',
                 '166-170cm', '171-175cm', '176-180cm', '181-185cm', '190cm 이상']
df_musinsa_reviews['user_height_group'] = pd.cut(
    df_musinsa_reviews['user_height'],
    bins=height_bins, labels=height_labels, right=True
)

# 8. user_weight_group: 실수값 → 5kg 구간 파생
weight_bins   = [0, 44, 49, 54, 59, 64, 69, 74, 79, 84, 89, float('inf')]
weight_labels = ['44kg 이하', '45-49kg', '50-54kg', '55-59kg', '60-64kg',
                 '65-69kg', '70-74kg', '75-79kg', '80-84kg', '85-89kg', '90kg 이상']
df_musinsa_reviews['user_weight_group'] = pd.cut(
    df_musinsa_reviews['user_weight'],
    bins=weight_bins, labels=weight_labels, right=True
)

# 9. 스키마 순서로 컬럼 재배치
schema_cols = [
    'collect_date', 'platform', 'search_keyword',
    'review_id', 'product_id', 'product_name',
    'review_date', 'year', 'month', 'review_body', 'rating',
    'helpful_count', 'has_image', 'purchase_option',
    'user_height', 'user_weight', 'user_height_group', 'user_weight_group',
    'product_url',
]
df_musinsa_reviews = df_musinsa_reviews[schema_cols]

# 10. 타입 지정
df_musinsa_reviews['review_id']         = df_musinsa_reviews['review_id'].astype(str)
df_musinsa_reviews['product_id']        = df_musinsa_reviews['product_id'].astype(str)
df_musinsa_reviews['search_keyword']    = df_musinsa_reviews['search_keyword'].astype(str)
df_musinsa_reviews['rating']            = df_musinsa_reviews['rating'].astype(int)
df_musinsa_reviews['helpful_count']     = df_musinsa_reviews['helpful_count'].astype(int)
df_musinsa_reviews['has_image']         = df_musinsa_reviews['has_image'].astype(int)
df_musinsa_reviews['user_height']       = df_musinsa_reviews['user_height'].astype(float)
df_musinsa_reviews['user_weight']       = df_musinsa_reviews['user_weight'].astype(float)
df_musinsa_reviews['platform']          = df_musinsa_reviews['platform'].astype('category')
df_musinsa_reviews['purchase_option']   = df_musinsa_reviews['purchase_option'].astype('category')
df_musinsa_reviews['user_height_group'] = df_musinsa_reviews['user_height_group'].astype('category')
df_musinsa_reviews['user_weight_group'] = df_musinsa_reviews['user_weight_group'].astype('category')

print(f"최종 shape: {df_musinsa_reviews.shape}")
print()
print(df_musinsa_reviews.dtypes)
print()
df_musinsa_reviews.head(3)

최종 shape: (1944, 19)

collect_date         datetime64[us]
platform                   category
search_keyword                  str
review_id                       str
product_id                      str
product_name                    str
review_date          datetime64[us]
year                          int64
month                         int64
review_body                     str
rating                        int64
helpful_count                 int64
has_image                     int64
purchase_option            category
user_height                 float64
user_weight                 float64
user_height_group          category
user_weight_group          category
product_url                     str
dtype: object



,collect_date,platform,search_keyword,review_id,product_id,product_name,review_date,year,month,review_body,rating,helpful_count,has_image,purchase_option,user_height,user_weight,user_height_group,user_weight_group,product_url
0,2026-04-21 22:57:48,무신사,5733085,83547450,5733085,심리스 와이드 헤드밴드 SOGY VPOR,2026-04-09,2026,4,룰루레몬 헤어밴드 좋습니다. 러닝할때 아주 잘 사용해요,4,0,1,Solar Grey/Vapor · One Size,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5733085#revie...
1,2026-04-21 22:57:48,무신사,5733085,83449647,5733085,심리스 와이드 헤드밴드 SOGY VPOR,2026-04-04,2026,4,땀이 많은 편이라 러닝 한번 하고나면 상의가 반쯤 젖을만큼 땀을 흘리는데 확실히 밴...,5,0,0,Solar Grey/Vapor · One Size,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5733085#revie...
2,2026-04-21 22:57:48,무신사,5733085,83439220,5733085,심리스 와이드 헤드밴드 SOGY VPOR,2026-04-04,2026,4,안벗겨지고 깔끔하고 무난합니다. 땀도 잘 흡수하고 가볍고 빨리 마르고 착용감도 너어...,5,0,1,Solar Grey/Vapor · One Size,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5733085#revie...


In [12]:
# ─── 결측치 비율 확인 ─────────────────────────────────────────────────────────

null_df = pd.DataFrame({
    '결측치 수': df_musinsa_reviews.isnull().sum(),
    '결측치 비율(%)': (df_musinsa_reviews.isnull().sum() / len(df_musinsa_reviews) * 100).round(2),
})
null_df[null_df['결측치 수'] > 0].sort_values('결측치 비율(%)', ascending=False)

,결측치 수,결측치 비율(%)
user_height,344,17.7
user_weight,344,17.7
user_height_group,344,17.7
user_weight_group,344,17.7


In [13]:
# ─── 중복 확인 ───────────────────────────────────────────────────────────────

dup_id = df_musinsa_reviews.duplicated(subset='review_id', keep=False)
print(f"review_id 중복: {dup_id.sum()}건 ({dup_id.sum() / len(df_musinsa_reviews) * 100:.2f}%)")

dup_content = df_musinsa_reviews.duplicated(subset=['product_id', 'review_body'], keep=False)
print(f"product_id + review_body 중복: {dup_content.sum()}건 ({dup_content.sum() / len(df_musinsa_reviews) * 100:.2f}%)")

if dup_id.sum() > 0:
    print()
    print("▶ review_id 중복 샘플")
    display(df_musinsa_reviews[dup_id].sort_values('review_id').head(6))

review_id 중복: 479건 (24.64%)
product_id + review_body 중복: 601건 (30.92%)

▶ review_id 중복 샘플


,collect_date,platform,search_keyword,review_id,product_id,product_name,review_date,year,month,review_body,rating,helpful_count,has_image,purchase_option,user_height,user_weight,user_height_group,user_weight_group,product_url
1233,2026-04-21 23:12:13,무신사,5895638,81780441,5895638,"에브 투 스트리트 크롭 레이서백 탱크탑 *라이트 서포트, B C컵",2026-01-21,2026,1,역시 룰루레몬 편하네요!! 너무저렴하게 잘샀어요~~~~~,5,0,0,Graphite Grey · 6,168.0,60.0,166-170cm,60-64kg,https://www.musinsa.com/products/5895638#revie...
1669,2026-04-21 23:27:52,무신사,5895638,81780441,5895638,"에브 투 스트리트 크롭 레이서백 탱크탑 *라이트 서포트, B C컵",2026-01-21,2026,1,역시 룰루레몬 편하네요!! 너무저렴하게 잘샀어요~~~~~,5,0,0,Graphite Grey · 6,168.0,60.0,166-170cm,60-64kg,https://www.musinsa.com/products/5895638#revie...
1868,2026-04-21 23:37:11,무신사,5895638,81780441,5895638,"에브 투 스트리트 크롭 레이서백 탱크탑 *라이트 서포트, B C컵",2026-01-21,2026,1,역시 룰루레몬 편하네요!! 너무저렴하게 잘샀어요~~~~~,5,0,0,Graphite Grey · 6,168.0,60.0,166-170cm,60-64kg,https://www.musinsa.com/products/5895638#revie...
1894,2026-04-21 23:37:17,무신사,5895638,81780441,5895638,"에브 투 스트리트 크롭 레이서백 탱크탑 *라이트 서포트, B C컵",2026-01-21,2026,1,역시 룰루레몬 편하네요!! 너무저렴하게 잘샀어요~~~~~,5,0,0,Graphite Grey · 6,168.0,60.0,166-170cm,60-64kg,https://www.musinsa.com/products/5895638#revie...
1920,2026-04-21 23:37:23,무신사,5895638,81780441,5895638,"에브 투 스트리트 크롭 레이서백 탱크탑 *라이트 서포트, B C컵",2026-01-21,2026,1,역시 룰루레몬 편하네요!! 너무저렴하게 잘샀어요~~~~~,5,0,0,Graphite Grey · 6,168.0,60.0,166-170cm,60-64kg,https://www.musinsa.com/products/5895638#revie...
1519,2026-04-21 23:21:50,무신사,5900413,81793794,5900413,"플로우 Y 브라 Nulu *라이트 서포트, B C컵 아시아 핏",2026-01-22,2026,1,일상생활할때 입으려면 한 사이즈씩은 업해야 할 것 같아요 핏은 엄청 이쁩니다,5,0,0,Light Ivory · S,157.0,53.0,156-160cm,50-54kg,https://www.musinsa.com/products/5900413#revie...


In [14]:
# ─── 중복 제거 ───────────────────────────────────────────────────────────────
# review_id 중복은 재수집으로 인한 크롤링 중복 → review_id 기준 제거

before = len(df_musinsa_reviews)
df_musinsa_reviews = df_musinsa_reviews.drop_duplicates(subset='review_id', keep='first').reset_index(drop=True)
after_id = len(df_musinsa_reviews)
print(f"[1] review_id 중복 제거: {before:,} → {after_id:,}건 (제거: {before - after_id}건)")

# 잔여 내용 중복 재확인
dup_content = df_musinsa_reviews.duplicated(subset=['product_id', 'review_body'], keep=False)
print(f"[2] 잔여 product_id + review_body 중복: {dup_content.sum()}건")

if dup_content.sum() > 0:
    display(df_musinsa_reviews[dup_content].sort_values(['product_id', 'review_body']).head(6))

[1] review_id 중복 제거: 1,944 → 1,581건 (제거: 363건)
[2] 잔여 product_id + review_body 중복: 126건


,collect_date,platform,search_keyword,review_id,product_id,product_name,review_date,year,month,review_body,rating,helpful_count,has_image,purchase_option,user_height,user_weight,user_height_group,user_weight_group,product_url
189,2026-04-21 22:58:24,무신사,5732069,83550899,5732069,소프트 저지 클래식 핏 팬츠 *레귤러 BLK,2026-04-09,2026,4,배송이 매우 빠르고 바지도 마음에 드네요 잘 입을게요,5,0,0,Black · M,170.0,75.0,166-170cm,75-79kg,https://www.musinsa.com/products/5732069#revie...
190,2026-04-21 22:58:24,무신사,5732069,83550894,5732069,소프트 저지 클래식 핏 팬츠 *레귤러 BLK,2026-04-09,2026,4,배송이 매우 빠르고 바지도 마음에 드네요 잘 입을게요,5,0,0,Black · M,170.0,75.0,166-170cm,75-79kg,https://www.musinsa.com/products/5732069#revie...
510,2026-04-21 23:00:33,무신사,5732140,81985487,5732140,메탈 벤트 테크 숏슬리브 셔츠 WHWH,2026-01-31,2026,1,소재나 착용감은 정말 좋구요. 다만 사이즈가 저한테 조금 애매 하네요. 좀 더 여유...,4,0,1,White · L,182.0,87.0,181-185cm,85-89kg,https://www.musinsa.com/products/5732140#revie...
517,2026-04-21 23:00:33,무신사,5732140,81985484,5732140,메탈 벤트 테크 숏슬리브 셔츠 WHWH,2026-01-31,2026,1,소재나 착용감은 정말 좋구요. 다만 사이즈가 저한테 조금 애매 하네요. 좀 더 여유...,4,0,1,White · L,182.0,87.0,181-185cm,85-89kg,https://www.musinsa.com/products/5732140#revie...
733,2026-04-21 23:03:32,무신사,5732299,82249493,5732299,페이스 브레이커 팬츠 BLK,2026-02-13,2026,2,편안한 착용감에 기능성 소재가 만족스러워요. 다만 페이스브레이커 쇼츠에 비해서는 다...,5,0,1,Black · XL,180.0,80.0,176-180cm,80-84kg,https://www.musinsa.com/products/5732299#revie...
736,2026-04-21 23:03:32,무신사,5732299,82249508,5732299,페이스 브레이커 팬츠 BLK,2026-02-13,2026,2,편안한 착용감에 기능성 소재가 만족스러워요. 다만 페이스브레이커 쇼츠에 비해서는 다...,5,0,1,Black · XL,180.0,80.0,176-180cm,80-84kg,https://www.musinsa.com/products/5732299#revie...


In [15]:
# ─── 잔여 내용 중복 샘플 상세 확인 ──────────────────────────────────────────

dup_mask = df_musinsa_reviews.duplicated(subset=['product_id', 'review_body'], keep=False)
dup_sample = (
    df_musinsa_reviews[dup_mask]
    .sort_values(['product_id', 'review_body'])
    [['review_id', 'product_id', 'product_name', 'review_date', 'review_body', 'rating']]
)

print(f"중복 그룹 수: {dup_sample.groupby(['product_id','review_body']).ngroups}개")
print(f"중복 포함 총 행 수: {len(dup_sample)}건")
print()
display(dup_sample.head(12))

중복 그룹 수: 63개
중복 포함 총 행 수: 126건



,review_id,product_id,product_name,review_date,review_body,rating
189,83550899,5732069,소프트 저지 클래식 핏 팬츠 *레귤러 BLK,2026-04-09,배송이 매우 빠르고 바지도 마음에 드네요 잘 입을게요,5
190,83550894,5732069,소프트 저지 클래식 핏 팬츠 *레귤러 BLK,2026-04-09,배송이 매우 빠르고 바지도 마음에 드네요 잘 입을게요,5
510,81985487,5732140,메탈 벤트 테크 숏슬리브 셔츠 WHWH,2026-01-31,소재나 착용감은 정말 좋구요. 다만 사이즈가 저한테 조금 애매 하네요. 좀 더 여유...,4
517,81985484,5732140,메탈 벤트 테크 숏슬리브 셔츠 WHWH,2026-01-31,소재나 착용감은 정말 좋구요. 다만 사이즈가 저한테 조금 애매 하네요. 좀 더 여유...,4
733,82249493,5732299,페이스 브레이커 팬츠 BLK,2026-02-13,편안한 착용감에 기능성 소재가 만족스러워요. 다만 페이스브레이커 쇼츠에 비해서는 다...,5
736,82249508,5732299,페이스 브레이커 팬츠 BLK,2026-02-13,편안한 착용감에 기능성 소재가 만족스러워요. 다만 페이스브레이커 쇼츠에 비해서는 다...,5
1371,81931322,5732335,페이스 브레이커 팬츠 GGRE,2026-01-29,룰루레몬은 항상 대만족이고 실망시키질 않네요~ 좋은가격에 잘 구매 했습니다~,5
1372,81931315,5732335,페이스 브레이커 팬츠 GGRE,2026-01-29,룰루레몬은 항상 대만족이고 실망시키질 않네요~ 좋은가격에 잘 구매 했습니다~,5
379,83359325,5732356,제로드 인 숏슬리브 셔츠 RWLI,2026-04-01,"여름에 출근용, 운동용, 러닝용, 등산용으로 구입했습니다. 가볍고 촉감이 너무 좋네요",5
387,83359310,5732356,제로드 인 숏슬리브 셔츠 RWLI,2026-04-01,"여름에 출근용, 운동용, 러닝용, 등산용으로 구입했습니다. 가볍고 촉감이 너무 좋네요",5


In [16]:
# ─── 내용 중복 제거 (product_id + review_body 기준, 첫 번째 유지) ─────────────

before = len(df_musinsa_reviews)
df_musinsa_reviews = df_musinsa_reviews.drop_duplicates(subset=['product_id', 'review_body'], keep='first').reset_index(drop=True)
after = len(df_musinsa_reviews)

print(f"내용 중복 제거: {before:,} → {after:,}건 (제거: {before - after}건)")
print(f"최종 shape: {df_musinsa_reviews.shape}")

내용 중복 제거: 1,581 → 1,518건 (제거: 63건)
최종 shape: (1518, 19)


In [ ]:
# # ─── 파일 저장 ───────────────────────────────────────────────────────────────

# output_path = './final_data/musinsa_reviews_cleaned_20260429.csv'
# df_musinsa_reviews.to_csv(output_path, index=False, encoding='utf-8-sig')
# print(f"저장 완료: {output_path} ({len(df_musinsa_reviews):,}건)")

저장 완료: ./data/musinsa_reviews_cleaned_20260425.csv (1,518건)


In [6]:
df = pd.read_csv('./final_data/lululemon_homepage_review_final_s2024.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11819 entries, 0 to 11818
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   collect_date       11819 non-null  str    
 1   platform           11819 non-null  str    
 2   search_keyword     11819 non-null  str    
 3   review_id          11819 non-null  int64  
 4   product_id         11819 non-null  str    
 5   product_name       10292 non-null  str    
 6   review_date        11819 non-null  str    
 7   year               11819 non-null  int64  
 8   month              11819 non-null  int64  
 9   review_body        11817 non-null  str    
 10  rating             11819 non-null  int64  
 11  helpful_count      11819 non-null  int64  
 12  has_image          0 non-null      float64
 13  purchase_option    4489 non-null   str    
 14  user_height        0 non-null      float64
 15  user_weight        0 non-null      float64
 16  user_height_group  282 non-null  

In [7]:
df.head()

,collect_date,platform,search_keyword,review_id,product_id,product_name,review_date,year,month,review_body,rating,helpful_count,has_image,purchase_option,user_height,user_weight,user_height_group,user_weight_group,product_url
0,2026-04-17,자사몰,prod11710026,384702382,prod11710026,메탈 벤트 테크 숏슬리브 셔츠,2026-04-16,2026,4,부드러운 옷 더운날 입기좋은 재질 부드러운 재질 땀냄새 나지 않는 재질 가격이 사...,5,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-04-17,자사몰,prod11710026,384590413,prod11710026,메탈 벤트 테크 숏슬리브 셔츠,2026-04-15,2026,4,같은옷 3개째 같은 색상의. 옷을 3개 구매 \n\n장점 : 오늘은 뭐입지 라는 고...,5,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-04-17,자사몰,prod11710026,384591246,prod11710026,메탈 벤트 테크 숏슬리브 셔츠,2026-04-15,2026,4,3점 두께감이있어서 호불호가있고 색감이화면과는 조금상이합니다. 불만족하였는데 리뷰...,3,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-04-17,자사몰,prod11710026,384333663,prod11710026,메탈 벤트 테크 숏슬리브 셔츠,2026-04-12,2026,4,운동할때 입기 편해요 색도 맘에 들고 운동할때 땀 흡수도 잘 되고 무엇보다 편합니다...,5,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-04-17,자사몰,prod11710026,383421185,prod11710026,메탈 벤트 테크 숏슬리브 셔츠,2026-04-04,2026,4,너무 좋아요 러닝을 하기위해서 샀는데 너무 좋습니다 굉장히 쾌적하고 러닝후 냄새도 ...,5,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# 컬럼 삭제
df = df.drop(columns=['search_keyword'])

In [14]:
# ─── review_body 정제: 띄어쓰기 제외 5자 이하 제거 ──────────────────────────

before = len(df)
mask = df['review_body'].fillna('').str.replace(' ', '', regex=False).str.len() > 5
df = df[mask].reset_index(drop=True)
after = len(df)

print(f'제거 전: {before:,}건 → 제거 후: {after:,}건 (제거: {before - after}건)')
print(f'review_body NaN: {df["review_body"].isna().sum()}건')

# review_body : content으로 컬럼명 변경
df = df.rename(columns={'review_body': 'content'})

제거 전: 11,817건 → 제거 후: 11,817건 (제거: 0건)
review_body NaN: 0건


In [15]:
# ─── purchase_option 파싱: size / color 분리 ─────────────────────────────────
# 무신사: 'Color · Size' 형식 / 자사몰: 'Size' 단일값

def parse_option(val):
    if pd.isna(val):
        return pd.NA, pd.NA
    parts = str(val).split('·')
    if len(parts) == 2:
        return parts[0].strip(), parts[1].strip()
    return pd.NA, val.strip()  # 단일값이면 사이즈만 존재, color는 NA

df[['purchase_option_color', 'purchase_option_size']] = df['purchase_option'].apply(
    lambda x: pd.Series(parse_option(x))
)

print('purchase_option_size 고유값:')
print(df['purchase_option_size'].value_counts(dropna=False))
print()
print('purchase_option_color 고유값:')
print(df['purchase_option_color'].value_counts(dropna=False))

purchase_option_size 고유값:
purchase_option_size
NaN    7328
S      1552
M      1375
L       652
XS      530
XL      258
2XL     103
XXS      11
3XL       8
Name: count, dtype: int64

purchase_option_color 고유값:
purchase_option_color
<NA>    7328
NaN     4489
Name: count, dtype: int64


In [16]:
# ─── women_size_yn: 사이즈에 w가 붙은 경우 1, 아니면 0 ──────────────────────

df['women_size_yn'] = df['purchase_option_size'].apply(
    lambda x: 1 if pd.notna(x) and str(x).strip().upper().startswith('W') else 0
)

print('women_size_yn 분포:')
print(df['women_size_yn'].value_counts())
print()
print('최종 shape:', df.shape)
df[['purchase_option', 'purchase_option_size', 'purchase_option_color', 'women_size_yn']].dropna(subset=['purchase_option']).head(10)

women_size_yn 분포:
women_size_yn
0    11817
Name: count, dtype: int64

최종 shape: (11817, 21)


,purchase_option,purchase_option_size,purchase_option_color,women_size_yn
12,M,M,NaN,0
13,L,L,NaN,0
14,L,L,NaN,0
16,2XL,2XL,NaN,0
17,2XL,2XL,NaN,0
21,XL,XL,NaN,0
22,L,L,NaN,0
23,L,L,NaN,0
24,XL,XL,NaN,0
26,XL,XL,NaN,0


In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11817 entries, 0 to 11816
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   collect_date           11817 non-null  str    
 1   platform               11817 non-null  str    
 2   review_id              11817 non-null  int64  
 3   product_id             11817 non-null  str    
 4   product_name           10291 non-null  str    
 5   review_date            11817 non-null  str    
 6   year                   11817 non-null  int64  
 7   month                  11817 non-null  int64  
 8   content                11817 non-null  str    
 9   rating                 11817 non-null  int64  
 10  helpful_count          11817 non-null  int64  
 11  has_image              0 non-null      float64
 12  purchase_option        4489 non-null   str    
 13  user_height            0 non-null      float64
 14  user_weight            0 non-null      float64
 15  user_height_g

In [18]:
df['purchase_option_size'].value_counts()

purchase_option_size
S      1552
M      1375
L       652
XS      530
XL      258
2XL     103
XXS      11
3XL       8
Name: count, dtype: int64

In [19]:
# ========================
# 최종 컬럼 정리
# ========================

# 1) 데이터 형변환
def convert_dtypes(df):
    # 날짜 컬럼
    for col in ["collect_date", "review_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # ID 컬럼
    for col in ["review_id", "product_id"]:
        if col in df.columns:
            df[col] = df[col].astype("string")

    # 문자열 컬럼
    for col in ["product_name", "content", "purchase_option", "product_url"]:
        if col in df.columns:
            df[col] = df[col].astype("string")

    # category 컬럼
    for col in ["platform", "purchase_option_color", "purchase_option_size", "user_height_group", "user_weight_group"]:
        if col in df.columns:
            df[col] = df[col].astype("category")

    # 정수형 컬럼
    for col, dtype in [("year", "Int16"), ("month", "Int8"), ("rating", "Int8"), ("helpful_count", "Int32")]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype(dtype)

    # 0/1 플래그 컬럼
    for col in ["has_image", "women_size_yn"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int8")

    # 실수형 컬럼
    for col in ["user_height", "user_weight"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")

    return df

# 사용
df = convert_dtypes(df)
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 11817 entries, 0 to 11816
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   collect_date           11817 non-null  datetime64[us]
 1   platform               11817 non-null  category      
 2   review_id              11817 non-null  string        
 3   product_id             11817 non-null  string        
 4   product_name           10291 non-null  string        
 5   review_date            11817 non-null  datetime64[us]
 6   year                   11817 non-null  Int16         
 7   month                  11817 non-null  Int8          
 8   content                11817 non-null  string        
 9   rating                 11817 non-null  Int8          
 10  helpful_count          11817 non-null  Int32         
 11  has_image              11817 non-null  int8          
 12  purchase_option        4489 non-null   string        
 13  user_height 

In [20]:
# ======================
# 컬럼 순서 재정렬
# ======================

column_order = [
    'collect_date',
    'platform',
    'review_id',
    'product_id',
    'product_name',
    'review_date',
    'year',
    'month',
    'content',
    'rating',
    'helpful_count',
    'has_image',
    'purchase_option',
    'purchase_option_color',
    'purchase_option_size',
    'women_size_yn',
    'user_height',
    'user_weight',
    'user_height_group',
    'user_weight_group',
    'product_url'
]

# 컬럼 존재하는 것만 선택 (안전하게)
df = df[
    [col for col in column_order if col in df.columns]
]

# 확인
df.head()

,collect_date,platform,review_id,product_id,product_name,review_date,year,month,content,rating,helpful_count,has_image,purchase_option,purchase_option_color,purchase_option_size,women_size_yn,user_height,user_weight,user_height_group,user_weight_group,product_url
0,2026-04-17,자사몰,384702382,prod11710026,메탈 벤트 테크 숏슬리브 셔츠,2026-04-16,2026,4,부드러운 옷 더운날 입기좋은 재질 부드러운 재질 땀냄새 나지 않는 재질 가격이 사...,5,0,0,<NA>,NaN,NaN,0,NaN,NaN,NaN,NaN,<NA>
1,2026-04-17,자사몰,384590413,prod11710026,메탈 벤트 테크 숏슬리브 셔츠,2026-04-15,2026,4,같은옷 3개째 같은 색상의. 옷을 3개 구매 \n\n장점 : 오늘은 뭐입지 라는 고...,5,0,0,<NA>,NaN,NaN,0,NaN,NaN,NaN,NaN,<NA>
2,2026-04-17,자사몰,384591246,prod11710026,메탈 벤트 테크 숏슬리브 셔츠,2026-04-15,2026,4,3점 두께감이있어서 호불호가있고 색감이화면과는 조금상이합니다. 불만족하였는데 리뷰...,3,0,0,<NA>,NaN,NaN,0,NaN,NaN,NaN,NaN,<NA>
3,2026-04-17,자사몰,384333663,prod11710026,메탈 벤트 테크 숏슬리브 셔츠,2026-04-12,2026,4,운동할때 입기 편해요 색도 맘에 들고 운동할때 땀 흡수도 잘 되고 무엇보다 편합니다...,5,0,0,<NA>,NaN,NaN,0,NaN,NaN,NaN,NaN,<NA>
4,2026-04-17,자사몰,383421185,prod11710026,메탈 벤트 테크 숏슬리브 셔츠,2026-04-04,2026,4,너무 좋아요 러닝을 하기위해서 샀는데 너무 좋습니다 굉장히 쾌적하고 러닝후 냄새도 ...,5,0,0,<NA>,NaN,NaN,0,NaN,NaN,NaN,NaN,<NA>


In [21]:
# 데이터 저장
df.to_csv('./final_data/lululemon_homepage_review_final_s2024.csv', index=False, encoding='utf-8-sig')